# Setup influx connection

In [2]:
import influxdb_client

In [3]:
# Setup e connessione al server InfluxDB
org_name = "spamic"
bucket_name = "sebd"
token = "6ANpdrZyP8Nk7HBib5puP7dx8fJrFNG2xCPlBiXqDSKAsSfjKsUMPlgEVvoQMllyP0HCDAlherDBZZA_z42utw=="
url = "http://localhost:8086"

client = influxdb_client.InfluxDBClient(
    url=url,
    token=token,
    org=org_name
)

In [28]:
# Scrittura di un record
write_api = client.write_api()
write_api.write(
    bucket_name,
    org_name,
    ["h2o_feet,location=coyote_creek water_level=1"]
)
p = (
    influxdb_client.Point("h2o_feet") #measurement
    .tag("location", "Prague")
    .field("temperature", 300.3)
)

write_api.write(bucket=bucket_name, org=org_name, record=p)

### Select

In [4]:
# Select query
query = '''
from(bucket: "sebd")
  |> range(start: -10d)
  |> filter(fn: (r) => r._measurement == "h2o_feet")
'''
tables = client.query_api().query(query, org=org_name)
for table in tables:
    for record in table.records:
        print(record.get_field() + " " + str(record.get_value()))

temperature 300.3
temperature 300.3
water_level 1.0
water_level 300.3
water_level 1.0
water_level 1.0


#### Flux object
- get_measurement(): ritorna il nome della tabella (measurement) 
- get_field(): ritorna il nome della misura 
- get_value(): ritorna il valore letto 
- values: ritorna una mappa di valori per ogni colonna 
- values.get(<your_tag>): ritorna il valore della colonna indicata (tag) 
- get_time(): ritorna il timestamp del record 
- get_start(): ritorna il timestamp inferiore di tutti i record della tabella 
- get_stop(): ritorna il timestamp superiore di tutti i record della tabella

In [5]:
# Query with absolute temperal range
query = '''
from(bucket: "sebd")
  |> range(start: 2026-03-10T00:00:00Z, stop: 2026-03-15T00:00:00Z,)
  |> filter(fn: (r) => r._measurement == "h2o_feet")
'''
tables = client.query_api().query(query, org=org_name)
for table in tables:
    for record in table.records:
        print(record.get_field() + " " + str(record.get_value()) + " " + str(record.get_time()))

temperature 300.3 2026-03-12 14:49:17.550775+00:00
temperature 300.3 2026-03-12 14:47:45.067064+00:00
water_level 1.0 2026-03-12 14:14:25.886139+00:00
water_level 300.3 2026-03-12 14:16:09.714441+00:00
water_level 1.0 2026-03-12 14:47:45.067064+00:00
water_level 1.0 2026-03-12 14:49:17.550775+00:00


In [6]:
# Select query with field
query = '''
from(bucket: "sebd")
  |> range(start: -10d)
  |> filter(fn: (r) => r._field == "temperature")
  |> filter(fn: (r) => r._measurement == "h2o_feet")
'''
tables = client.query_api().query(query, org=org_name)
for table in tables:
    for record in table.records:
        print(record.get_field() + " " + str(record.get_value()))

temperature 300.3
temperature 300.3


In [7]:
# Select query with tag
query = '''
from(bucket: "sebd")
  |> range(start: -10d)
  |> filter(fn: (r) => r.location == "Prague")
  |> filter(fn: (r) => r._measurement == "h2o_feet")
'''
tables = client.query_api().query(query, org=org_name)
for table in tables:
    for record in table.records:
        print(record.get_field() + " " + str(record.get_value()) + " " + record.values.get("location"))

temperature 300.3 Prague


In [13]:
# Select query with aggregation
query = '''
from(bucket: "sebd")
  |> range(start: -10d)
  |> filter(fn: (r) => r.location == "Prague")
  |> filter(fn: (r) => r._measurement == "h2o_feet")
  |> mean()
'''
tables = client.query_api().query(query, org=org_name)
for table in tables:
    for record in table.records:
        print(record.get_field() + " " + str(record.get_value()))

temperature 300.3


### Delete record

In [12]:
from datetime import datetime

# get delete API
delete_api = client.delete_api()
bucket_name = "sebd"
start = "1900-01-01T00:00:00Z" # Starting date
stop = datetime.utcnow().isoformat() + "Z"
predicate = "_measurements = h2o_feet"

# delete
delete_api.delete(start, stop, predicate, bucket_name, org_name)